In [0]:
%pip install openpyxl

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python or dbutils.library.restartPython()

In [0]:
import json
import pandas as pd

# 1. Read the uploaded small data jobs excel file
df_jobs = pd.read_excel("/Volumes/workspace/default/mum/data_jobs_20rows.xlsx")

# Clean column names and fill missing values foer streaming
df_jobs = df_jobs.fillna("")
# Define he landing folder path in a Unity Catalog Volume (DBFS FileStore is disabled)
landing_folder = "/Volumes/workspace/default/mum/datajobs_stream/landing_json"

# Clear old landing files if any
try:
    dbutils.fs.rm(landing_folder, recurse=True)
except:
    pass

#3. Simulate streaming ingestion by writing each roww(or chunks of rows) as a separate JSON file   
print("Dropping simulated job posting streams into landing folder...")
for idx, row in df_jobs.iterrows():
    record = row.to_dict()
    # Convert date to string fomrat if necessary
    record["job_posted_date"] = str(record["job_posted_date"])

    # Write each job posting as sa timestamped JSON file
    filename = f"{landing_folder}/job_posting_{idx:03d}.json"
    dbutils.fs.put(filename, json.dumps(record), overwrite=True)

print(
    f"Successfully dropped {len(df_jobs)} simulated JSON job records into"
    f" {landing_folder}!"
)
    

Dropping simulated job posting streams into landing folder...
Wrote 406 bytes.
Wrote 364 bytes.
Wrote 330 bytes.
Wrote 322 bytes.
Wrote 349 bytes.
Wrote 365 bytes.
Wrote 369 bytes.
Wrote 322 bytes.
Wrote 360 bytes.
Wrote 347 bytes.
Wrote 353 bytes.
Wrote 336 bytes.
Wrote 334 bytes.
Wrote 438 bytes.
Wrote 327 bytes.
Wrote 349 bytes.
Wrote 400 bytes.
Wrote 344 bytes.
Wrote 353 bytes.
Wrote 363 bytes.
Successfully dropped 20 simulated JSON job records into /Volumes/workspace/default/mum/datajobs_stream/landing_json!


Run the Spark Structured Streaming Pipeline

In [0]:
from pyspark.sql.functions import col, current_timestamp, expr, when
from pyspark.sql.types import DoubleType, StringType, StructField, StructType

# Define storage paths
checkpoint_dir = "/Volumes/workspace/default/mum/datajobs_stream/checkpoints_jobs"
delta_sink_path = "/Volumes/workspace/default/mum/datajobs_stream/delta_jobs_sink"

# Clean up old sink and checkpoints to start fresh
dbutils.fs.rm(delta_sink_path, recurse=True)
dbutils.fs.rm(checkpoint_dir, recurse=True)

# Define schema matching the data_jobs structure
job_schema = StructType([
    StructField("job_title_short", StringType(), True),
    StructField("job_posted_date", StringType(), True),
    StructField("job_location", StringType(), True),
    StructField("job_country", StringType(), True),
    StructField("company_name", StringType(), True),
    StructField("job_via", StringType(), True),
    StructField("job_schedule_type", StringType(), True),
    StructField("salary_year_avg", DoubleType(), True),
    StructField("job_work_from_home", StringType(), True),
    StructField("job_skills", StringType(), True),
])

# 1. INGESTION: Read Stream from landing folder
raw_job_stream = (
    spark.readStream.schema(job_schema)
    .option("maxFilesPerTrigger", 2)  # Process 2 JSON files per micro-batch
    .json(landing_folder)
)

# 2. TRANSFORMATION: Clean and add business logic (e.g., salary tiers)
transformed_job_stream = (
    raw_job_stream.withColumn(
        "monthly_est_salary", expr("ROUND(salary_year_avg / 12, 2)")
    )
    .withColumn(
        "compensation_tier",
        when(col("salary_year_avg") >= 120000, "High-Comp Senior")
        .when(col("salary_year_avg") > 0, "Standard")
        .otherwise("Unpaid/Not Listed"),
    )
    .withColumn("stream_processed_at", current_timestamp())
)

# 3. STREAM SINK: Write continuously to Delta table
job_streaming_query = (
    transformed_job_stream.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_dir)
    .trigger(availableNow=True)  # Processes all currently available files once (continuous polling triggers are not supported on this serverless compute)
    .start(delta_sink_path)
)

print("Job streaming pipeline is active and monitoring landing folder!")

Job streaming pipeline is active and monitoring landing folder!


In [0]:
# Read the resulting Delta Lake sink
df_sink_result = spark.read.format("delta").load(delta_sink_path)

print(f"Total processed streaming records: {df_sink_result.count()}")
display(
    df_sink_result.select(
        "job_title_short",
        "company_name",
        "salary_year_avg",
        "monthly_est_salary",
        "compensation_tier",
        "stream_processed_at",
    )
)

Total processed streaming records: 20


job_title_short,company_name,salary_year_avg,monthly_est_salary,compensation_tier,stream_processed_at
Data Scientist,Standard Bank Group,70000.0,5833.33,Standard,2026-11-02T09:36:44.914Z
Data Scientist,Stripe,281450.5,23454.21,High-Comp Senior,2026-11-02T09:36:44.914Z
Senior Data Engineer,"Adroit Worldwide Media, Inc. (AWM Smart Shelf)",135000.0,11250.0,High-Comp Senior,2026-11-02T09:36:30.663Z
Data Analyst,Jobmatchingpartner Ltd,35000.0,2916.67,Standard,2026-11-02T09:36:30.663Z
Business Analyst,DISYS,null,null,Unpaid/Not Listed,2026-11-02T09:36:47.238Z
Data Scientist,Datalab USA,87500.0,7291.67,Standard,2026-11-02T09:36:47.238Z
Business Analyst,Denodo Technologies,90000.0,7500.0,Standard,2026-11-02T09:36:48.960Z
Data Scientist,Fortira Inc.,null,null,Unpaid/Not Listed,2026-11-02T09:36:48.960Z
Data Scientist,Walmart,195000.0,16250.0,High-Comp Senior,2026-11-02T09:36:43.729Z
Senior Data Scientist,Harnham,165000.0,13750.0,High-Comp Senior,2026-11-02T09:36:43.729Z
